In [27]:
from langchain.chat_models import init_chat_model
from tavily import TavilyClient
from dotenv import load_dotenv
import os 

# Load environment variables from .env file
load_dotenv()
tavily_client = TavilyClient(api_key="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")
model = init_chat_model(model="gpt-5-nano")

In [28]:
system_prompt = """
Sei un assistente personale in cucina. Aiuti gli utenti a trovare ricette basate sugli ingredienti che hanno a disposizione.
Se l'utente ti chiede una ricetta fornendo degli ingredienti, cerca sul web le ricette che utilizzano quegli ingredienti e fornisci una risposta con una ricetta dettagliata. 
Se non riesci a trovare una ricetta che corrisponde esattamente agli ingredienti forniti, suggerisci una ricetta simile e spiega quali ingredienti potrebbero essere sostituiti o omessi.
"""

In [29]:
from langchain.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver  
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any

@tool
def get_recipe(query: str) -> Dict[str, Any]:
    """Get a recipe from web based on the query."""
    return tavily_client.search(query=query)
    
agent = create_agent(
    model=model, 
    system_prompt=system_prompt,
    tools=[get_recipe], 
    checkpointer=InMemorySaver()
)


In [30]:
question = HumanMessage(content="Sto cercando una ricetta per la cena, ho del pollo e delle zucchine in frigo. Cosa posso cucinare?")
config = {"configurable": {"thread_id": "personal_chef_thread"}}
response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [31]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Sto cercando una ricetta per la cena, ho del pollo e delle zucchine in frigo. Cosa posso cucinare?', additional_kwargs={}, response_metadata={}, id='840104f5-8952-4d5d-a11f-caac13ed9d66'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 219, 'prompt_tokens': 275, 'total_tokens': 494, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DaM1gmdEmIDG9x4XPnv9ww5IAjXmo', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ddea4-fb2a-7191-9c80-c08b54cd79a3-0', tool_calls=[{'name': 'get_recipe', 'args': {'query': 'pollo zucchine ricetta'}, 'id': 'call_8U3J4qstxsApiPJajTrENGGV', 'type': 'tool_call'}], i

Per avviare l'IDE di langgraph: 
```bash 
langgraph dev
```